# 03. Exploratory Data Analysis & Temporal Traffic Aggregation
This notebook aggregates both datasets into 1-minute intervals (requests/minute) to allow a scientifically fair comparison of normal vs. peak operational regimes. We also compute rolling statistics, temporal volume boxplots, and numeric correlations.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
import matplotlib.pyplot as plt
from feature_engineering import aggregate_ecommerce_to_minutes, aggregate_web_server_to_minutes
from eda import compare_regimes_statistics, analyze_correlations
from visualization import plot_traffic_over_time, plot_rolling_statistics, plot_temporal_patterns


## Load preprocessed datasets


In [ ]:
df_ec = pd.read_csv('../data/processed/clean_ecommerce_logs.csv')
df_ws = pd.read_csv('../data/processed/clean_web_server_logs.csv')
df_ec['accessed_date'] = pd.to_datetime(df_ec['accessed_date'])
df_ws['timestamp'] = pd.to_datetime(df_ws['timestamp'])


## Aggregate to 1-minute intervals
We construct a complete datetime minute index for both datasets to ensure 0-request minutes are included.


In [ ]:
df_ec_min, threshold_ec = aggregate_ecommerce_to_minutes(df_ec)
df_ws_min, threshold_ws = aggregate_web_server_to_minutes(df_ws)

print(f'E-Commerce 1-minute intervals: {len(df_ec_min)} mins (Peak operational threshold: {threshold_ec:.1f} req/min)')
print(f'Web Server 1-minute intervals: {len(df_ws_min)} mins (Peak operational threshold: {threshold_ws:.1f} req/min)')


## Compare Normal vs. Peak Regime Statistics
Peak traffic is operationally defined as observations at or above the 90th percentile (or at least 1.0 requests/minute for sparse data).


In [ ]:
ec_regimes = compare_regimes_statistics(df_ec_min, 'requests_per_minute')
ws_regimes = compare_regimes_statistics(df_ws_min, 'requests_per_minute')

print('=== E-Commerce Regime Comparison ===')
print(ec_regimes)
print('\n=== Web Server Regime Comparison ===')
print(ws_regimes)


## Correlation Analysis


In [ ]:
ec_num_cols = ['requests_per_minute', 'unique_ips', 'total_sales', 'avg_duration_sec', 'total_bytes', 'rolling_std']
ec_corr = analyze_correlations(df_ec_min, ec_num_cols)
print('E-Commerce correlation matrix:')
print(ec_corr.round(4))


## Visualizations
We'll plot the traffic over time, rolling averages, and hour/day of the week distributions.


In [ ]:
# Generate plots inline
%matplotlib inline

# 1. Plot Traffic Over Time
plot_traffic_over_time(df_ec_min, '../outputs/figures/ec_traffic_over_time.png', 'E-Commerce')
plot_traffic_over_time(df_ws_min, '../outputs/figures/ws_traffic_over_time.png', 'Web Server')

# 2. Plot Rolling Mean & Volatility
plot_rolling_statistics(df_ec_min, '../outputs/figures/ec_rolling_statistics.png', 'E-Commerce')

# 3. Plot Temporal Boxplots
plot_temporal_patterns(df_ec_min, '../outputs/figures/ec_temporal_patterns.png', 'E-Commerce')
